# 分散 Self-Play — Kaggle worker

試合を生成してシャードを1ファイル出すだけ。学習はしない。**GPU 不要・インターネット不要。**

入力: Dataset `ptcg-distributed-selfplay`(`ptcg_repo.zip` と `run_*.zip` を含む)
出力: `/kaggle/working/kaggle.npz` — Notebook の Output からダウンロードする

Save & Run All にすれば、ブラウザを閉じても裏で最後まで走る。


## 1. 環境確認と展開


In [ ]:
!python -V
!nproc


In [ ]:
import glob, os, shutil, zipfile

# まず入力に何が来ているかを必ず出す。Kaggle は Dataset にアップロードした zip を
# 展開して置くことがあるので、zip のままでも展開済みでも動くようにする。
print('--- /kaggle/input ---')
for p in sorted(glob.glob('/kaggle/input/**', recursive=True))[:60]:
    print(' ', p)

os.makedirs('/kaggle/temp/ptcg', exist_ok=True)

def _place(zip_glob, marker_glob, marker_depth, dest):
    """zip があれば展開、無ければ展開済みの場所を marker から探してコピー。"""
    z = glob.glob(zip_glob, recursive=True)
    if z:
        print('zip から展開:', z[0])
        zipfile.ZipFile(z[0]).extractall(dest)
        return
    m = glob.glob(marker_glob, recursive=True)
    if not m:
        raise SystemExit(f'入力が見つからない: {zip_glob} も {marker_glob} も無い。'
                         'Dataset が Notebook に添付されているか確認する。')
    root = m[0]
    for _ in range(marker_depth):
        root = os.path.dirname(root)
    print('展開済みをコピー:', root)
    shutil.copytree(root, dest, dirs_exist_ok=True)

_place('/kaggle/input/**/ptcg_repo.zip',
       '/kaggle/input/**/sample_submission/cg/libcg.so', 3, '/kaggle/temp/ptcg')
# run zip は中に run/ を含むので展開先は /kaggle/temp。展開済みの場合は run/ 自体をコピー。
if glob.glob('/kaggle/input/**/run_*.zip', recursive=True):
    _place('/kaggle/input/**/run_*.zip', '', 0, '/kaggle/temp')
else:
    _place('/nonexistent', '/kaggle/input/**/run/run.json', 1, '/kaggle/temp/run')


# 必要なものが本当に来ているかを名指しで確認する(Dataset の版が古いと欠ける)
need = ['kaggle_replays/rl/distributed/worker.py',
        'kaggle_replays/rl/collect_parallel.py',
        'sample_submission/cg/libcg.so',
        'kaggle_replays/meta_analysis/archetype_decks/dragapult_ex/01.csv']
missing = [n for n in need if not os.path.exists('/kaggle/temp/ptcg/' + n)]
print('\n--- 必要ファイルの確認 ---')
for n in need:
    print(('  OK  ' if n not in missing else '  なし '), n)
if missing:
    raise SystemExit('Dataset の中身が足りない(古い版が添付された可能性)。'
                     f'欠けているもの: {missing}')

!ls /kaggle/temp/run /kaggle/temp/run/models
!cat /kaggle/temp/run/run.json


## 2. ゲームエンジンが動くか


In [ ]:
%cd /kaggle/temp/ptcg/kaggle_replays/rl
!python test_rollout.py


## 3. 並列処理の起動方式を確定させる


In [ ]:
# Linux での並列処理の起動方式を確かめる。
#
# 手元(Windows)は spawn が既定なので検証済みだが、Linux の既定は fork で、
# 親が読み込み済みの cg エンジン(ネイティブライブラリ)を子が引き継ぐ。ここで
# 両方を実際に走らせて、どちらが使えるかを確定させる。
#
# 2026-08-13 修正1: init_run.py の --learner-arch 既定値(dragapult_ex)の重み
# policy_weights_dragapult_ex.json は旧166次元で、現行715次元エンコーダと合わず
# PolicyModel.is_ready=False になる。is_ready=False のまま multiprocessing.Pool を
# 初期化すると initializer が例外で落ち続け、Pool が子プロセスを無限に再spawnして
# ハングする(実測: --timeout 1800 で TimeoutExpired、Kaggle 側は31分後に ERROR 終了)。
# 診断は「起動方式が動くか」だけ確かめればよいので、715次元と確認済みの
# alakazam_ctl_deck06_s42 は git 未コミットのため push_kaggle.py の git archive に
# 含まれず Kaggle 側で見つからない(2026-08-13 に実際に踏んだ)。git管理下で確実に
# 存在する production policy_weights.json(715次元)を代わりに使う。
# production既定(相手側、policy_weights.json も715次元)に
# 固定する。本番 run の内容とは無関係(あくまで診断用)。
#
# 2026-08-13 修正2: 上の修正だけでは再発した。原因は git archive HEAD が**コミット済み**
# の中身しか含めないこと。sample_submission/ptcg_ai/learning/policy_weights.json を
# 166→715次元へローカルで更新したがコミットし忘れており、ローカルの動作確認は作業ツリー版
# (715次元)に対して行われたのに、push_kaggle.py が実際に送ったのは HEAD コミット版
# (166次元のまま)だった。これも上と全く同じ「initializer 失敗 → Pool 無限 respawn →
# 見た目だけハング」を引き起こす。push_kaggle.py 側に未コミット検知のガードを追加した
# (build_payload の _check_dirty_shipped_files)ので、今後は push 時点で気づける。
#
# 2026-08-13 修正3: 上の2つは「重みの中身が壊れている」ケースだったが、根本原因は
# multiprocessing.Pool が initializer の失敗を親プロセスに伝えず子プロセスを無限に
# 再spawnし続けること自体(2026-08-13 実測: 166次元の重みを渡すと90秒で321回respawn、
# 例外は一度も親に届かない)。collect_parallel.parallel_collect 側に fail-fast の
# 安全網を追加した: (1) Pool を作る前に親プロセスで PolicyModel.is_ready を確認する
# preflight チェック(既知の失敗モードをミリ秒〜数秒で検出)、(2) processes=1 の使い捨て
# Pool で initializer の完走を30秒のタイムアウト付きで確認する probe(未知の失敗モード向け
# の一般的な安全網)。これにより、今後 initializer が何らかの理由で失敗しても、
# 1800秒ではなく数秒〜高々30数秒でエラーとして表面化するようになった(ローカルで実証済み:
# 壊れた重みを渡すと0.0秒で RuntimeError)。そのため、このセルのタイムアウトも
# 1800→180秒に短縮した(健全なケースでの実測所要時間は spawn/fork 各8試合×2プロセスで
# 約11秒。180秒あれば safety net の30秒×2 + 実際の収集時間を十分にカバーできる)。
import subprocess, sys, os

os.chdir('/kaggle/temp/ptcg/kaggle_replays/rl/distributed')

_LEARNING_DIR = '/kaggle/temp/ptcg/sample_submission/ptcg_ai/learning'

# 診断用に試合数の少ない run を作る(本番の run とは別物)。715次元の重みを明示指定する
# (--learner-arch/--opponent-arch の既定値解決だけに頼ると166次元の古い重みを拾いうる)。
subprocess.run([sys.executable, 'init_run.py', '--run-id', 'diag',
                '--run-dir', '/kaggle/temp/diagrun', '--workers', 'w0',
                '--learner-arch', 'alakazam',
                '--learner-weights', f'{_LEARNING_DIR}/policy_weights.json',
                '--opponent-arch', 'alakazam',
                '--games-per-worker', '8'], check=True)

for method in ('spawn', 'fork'):
    print(f'\n===== start-method = {method} =====', flush=True)
    r = subprocess.run(
        [sys.executable, 'worker.py', '--run-dir', '/kaggle/temp/diagrun',
         '--worker-id', 'w0', '--workers', '2', '--start-method', method,
         '--out', f'/tmp/diag_{method}.npz'],
        capture_output=True, text=True, timeout=180)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print('--- stderr ---'); print(r.stderr[-2000:])
    print(f'>>> {method}: ' + ('OK' if r.returncode == 0 else f'NG (exit {r.returncode})'))


## 4. 本番の収集

シャードは `/kaggle/working/` へ直接書き出す(Notebook の Output になる)。


In [ ]:
%cd /kaggle/temp/ptcg/kaggle_replays/rl/distributed
!python worker.py --run-dir /kaggle/temp/run --worker-id kaggle --workers 4 \
    --start-method spawn --out /kaggle/working/kaggle.npz
!ls -lh /kaggle/working/
